# Cerebrovascular Accident Prognosis using Supervised Machine Learning Algorithms

**Authors:** Bhalashri Sethuraman & Niveditha S  
**Copyright:** SW-16851/2023, Copyright Office New Delhi  

This notebook trains and evaluates Random Forest, Decision Tree, XGBoost, KNN, and Logistic Regression for CVA prognosis.

**Dataset:** [Healthcare Stroke Dataset — Kaggle](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset)  
Place `healthcare-dataset-stroke-data.csv` in the `data/` folder before running.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

## 1. Load Dataset

In [ ]:
# If running on Google Colab, uncomment the lines below to mount Drive:
# from google.colab import drive
# drive.mount('/content/gdrive')
# df = pd.read_csv('/content/gdrive/MyDrive/brain-tumor/healthcare-dataset-stroke-data.xls')

# Local run:
df = pd.read_csv('../data/healthcare-dataset-stroke-data.csv')

df.head()

In [ ]:
df.value_counts()

## 2. NULL Value Analysis

In [ ]:
print(df.isna().sum())
df.isna().sum().plot.barh()
plt.title('Missing values per column')
plt.show()

In [ ]:
df.describe()

In [ ]:
df.info()

## 3. Pre-processing + EDA

In [ ]:
# Drop id column — no predictive value
df = df.drop(['id'], axis=1)

In [ ]:
# Gender analysis
print(df['gender'].value_counts())
# Only 1 instance of 'Other' — replace with Female to reduce dimensionality
df['gender'] = df['gender'].replace('Other', 'Female')
df['gender'].value_counts().plot(kind='pie', title='Gender Distribution')
plt.show()

In [ ]:
# Target feature — Stroke
print(df['stroke'].value_counts())
df['stroke'].value_counts().plot(kind='bar', color='goldenrod', title='Stroke Count')
plt.show()
print('% of people who actually got a stroke:',
      (df['stroke'].value_counts()[1] / df['stroke'].value_counts().sum()).round(3) * 100)

In [ ]:
# Hypertension Analysis
df['hypertension'].value_counts().plot(kind='bar', color='indianred', title='Hypertension')
plt.show()

In [ ]:
# Work type Analysis
print(df['work_type'].value_counts())
df['work_type'].value_counts().plot(kind='pie', title='Work Type Distribution')
plt.show()

In [ ]:
# Smoking status Analysis
print(df['smoking_status'].value_counts())
df['smoking_status'].value_counts().plot.bar(stacked=True, title='Smoking Status')
plt.show()
df['smoking_status'].value_counts().plot.pie(
    labels=['Never Smoked', 'Unknown', 'Formerly Smoked', 'Smokes'],
    colors=['mediumseagreen', 'khaki', 'lightskyblue', 'lightcoral'],
    explode=[0.1, 0, 0, 0],
    autopct='%.0f%%',
    figsize=(4, 4)
)
plt.show()

In [ ]:
# Residence type Analysis
print(df['Residence_type'].value_counts())
df['Residence_type'].value_counts().plot(kind='bar', title='Residence Type')
plt.show()

In [ ]:
# BMI Analysis
print('BMI null count:', df['bmi'].isnull().sum())
sns.histplot(data=df['bmi'])
plt.title('BMI Histogram')
plt.show()
sns.boxplot(data=df['bmi'])
plt.title('BMI Boxplot')
plt.show()

Q1 = df['bmi'].quantile(0.25)
Q3 = df['bmi'].quantile(0.75)
IQR = Q3 - Q1
da = (df['bmi'] < (Q1 - 1.5 * IQR)) | (df['bmi'] > (Q3 + 1.5 * IQR))
print('BMI outlier counts:\n', da.value_counts())
print('% NULL values in bmi:', df['bmi'].isna().sum() / len(df['bmi']) * 100)

In [ ]:
df_na = df.loc[df['bmi'].isnull()]
g = df_na['stroke'].sum()
h = df['stroke'].sum()
print('People who got stroke and their BMI is NA:', g)
print('People who got stroke and their BMI is given:', h)
print('% of people with stroke in NaN BMI values:', g / h * 100)
print('% instances who got stroke:', df['stroke'].sum() / len(df) * 100)

# Cannot drop nulls — 40 of the 201 null-BMI people had a stroke
# Cannot use mean — outliers present; use median instead
print('Median of BMI:', df['bmi'].median())
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

In [ ]:
# Age Analysis
sns.histplot(data=df['age'])
plt.title('Age Histogram')
plt.show()
sns.boxplot(data=df['age'])
plt.title('Age Boxplot')
plt.show()

In [ ]:
# Average Glucose Level Analysis
sns.histplot(data=df['avg_glucose_level'])
plt.title('Avg Glucose Level Histogram')
plt.show()
sns.boxplot(data=df['avg_glucose_level'])
plt.title('Avg Glucose Level Boxplot')
plt.show()

Q1 = df['avg_glucose_level'].quantile(0.25)
Q3 = df['avg_glucose_level'].quantile(0.75)
IQR = Q3 - Q1
da = (df['avg_glucose_level'] < (Q1 - 1.5 * IQR)) | (df['avg_glucose_level'] > (Q3 + 1.5 * IQR))
print('Avg glucose outlier counts:\n', da.value_counts())

In [ ]:
# Correlation Matrix
corrmat = df.corr(numeric_only=True)
f, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(corrmat, ax=ax, cmap='YlGnBu', linewidth=0.8, annot=True)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Heart disease and marital status analysis
print(df['heart_disease'].value_counts())
df['heart_disease'].value_counts().plot(kind='pie', title='Heart Disease')
plt.show()

print(df['ever_married'].value_counts())
df['ever_married'].value_counts().plot(kind='pie', title='Ever Married')
plt.show()

In [ ]:
# Cross analysis — features vs stroke target
for col in ['gender', 'work_type', 'smoking_status', 'Residence_type', 'heart_disease', 'ever_married']:
    sns.countplot(x='stroke', hue=col, data=df)
    plt.title(f'Stroke vs {col}')
    plt.show()

In [ ]:
# Distribution overview
fig, axs = plt.subplots(2, 3, figsize=(23, 12))
sns.histplot(df['gender'], kde=False, ax=axs[0, 0])
sns.histplot(df['age'], kde=False, ax=axs[0, 1])
sns.histplot(df['work_type'], kde=False, ax=axs[0, 2])
sns.histplot(df['avg_glucose_level'], kde=False, ax=axs[1, 0])
sns.histplot(df['bmi'], kde=False, ax=axs[1, 1])
sns.histplot(df['smoking_status'], kde=False, ax=axs[1, 2])
plt.show()

fig, axs = plt.subplots(1, 3, figsize=(20, 6))
sns.boxplot(x=df['age'], ax=axs[0])
sns.boxplot(x=df['avg_glucose_level'], ax=axs[1])
sns.boxplot(x=df['bmi'], ax=axs[2])
plt.show()

## 4. Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_col = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
le = LabelEncoder()
for col in categorical_col:
    df[col] = le.fit_transform(df[col])

for col in df.columns:
    if df[col].dtype != 'float64':
        print(f'{col} has unique values: {df[col].unique()}')

## 5. Dummy Variables & Oversampling

In [ ]:
# One-hot encoding for numeric-binary attributes
df[['hypertension', 'heart_disease', 'stroke']] = df[['hypertension', 'heart_disease', 'stroke']].astype(str)
df = pd.get_dummies(df, drop_first=True)
df.head()

In [ ]:
# RandomOverSampler — dataset is heavily imbalanced (~4.9% stroke)
from imblearn.over_sampling import RandomOverSampler

oversample = RandomOverSampler(sampling_strategy='minority')
X = df.drop(['stroke_1'], axis=1)
y = df['stroke_1']
X_over, y_over = oversample.fit_resample(X, y)

## 6. Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

s = StandardScaler()
df[['bmi', 'avg_glucose_level', 'age']] = s.fit_transform(df[['bmi', 'avg_glucose_level', 'age']])

In [ ]:
# SMOTE for further balancing
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy='minority')
X, y = smote.fit_resample(df.loc[:, df.columns != 'stroke_1'], df['stroke_1'])
print('Shape of X: {}'.format(X.shape))
print('Shape of y: {}'.format(y.shape))

## 7. Train-Test Split (80-20)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_over, y_over, test_size=0.20, random_state=42)
print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

## 8. Model Training

### 8.1 Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, accuracy_score, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score, roc_curve)

clf = DecisionTreeClassifier()
clf = clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_pred_prob_dt = clf.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_pred)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, linewidth=2, color='darkorange')
plt.plot([0, 1], [0, 1], 'r--')
plt.title('ROC Curve of DECISION TREE')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.show()

cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_).plot()
plt.show()

print('[INFO] evaluating after training...')
print('!!Confusion Matrix Report for Decision Tree!!')
print(classification_report(y_test, y_pred))
print('Accuracy_score:', '{:.4%}'.format(accuracy_score(y_test, y_pred)))
print('ROC AUC Score:', '{:.4%}'.format(roc_auc_score(y_test, y_pred_prob_dt)))

### 8.2 KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=2)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
y_pred_prob_knn = knn.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_pred_knn)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, linewidth=2, color='darkorange')
plt.plot([0, 1], [0, 1], 'r--')
plt.title('ROC Curve of KNN')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.show()

cm = confusion_matrix(y_test, y_pred_knn, labels=knn.classes_)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=knn.classes_).plot()
plt.show()

print('[INFO] evaluating after training...')
print('!!Confusion Matrix Report for KNN!!')
print(classification_report(y_test, y_pred_knn))
print('Accuracy_score:', '{:.4%}'.format(accuracy_score(y_test, y_pred_knn)))
print('ROC AUC Score:', '{:.4%}'.format(roc_auc_score(y_test, y_pred_prob_knn)))

### 8.3 XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier()
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
y_pred_prob_xgb = xgb.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_pred_prob_xgb)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, linewidth=2, color='darkorange')
plt.plot([0, 1], [0, 1], 'r--')
plt.title('ROC Curve of XGBOOST')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.show()

cm = confusion_matrix(y_test, y_pred_xgb, labels=xgb.classes_)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=xgb.classes_).plot()
plt.show()

print('[INFO] evaluating after training...')
print('!!Confusion Matrix Report for XGBoost!!')
print(classification_report(y_test, y_pred_xgb))
print('Accuracy_score:', '{:.4%}'.format(accuracy_score(y_test, y_pred_xgb)))
print('ROC AUC Score:', '{:.4%}'.format(roc_auc_score(y_test, y_pred_prob_xgb)))

### 8.4 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators=100)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
y_pred_prob_rf = rf_clf.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_pred_prob_rf)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, linewidth=2, color='darkorange')
plt.plot([0, 1], [0, 1], 'r--')
plt.title('ROC Curve of RANDOM FOREST')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.show()

cm = confusion_matrix(y_test, y_pred_rf, labels=rf_clf.classes_)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_clf.classes_).plot()
plt.show()

print('[INFO] evaluating after training...')
print('!!Confusion Matrix Report for Random Forest!!')
print(classification_report(y_test, y_pred_rf))
print('Accuracy_score:', '{:.4%}'.format(accuracy_score(y_test, y_pred_rf)))
print('ROC AUC Score:', '{:.4%}'.format(roc_auc_score(y_test, y_pred_prob_rf)))

### 8.5 Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(random_state=0)
classifier.fit(X_train, y_train)
y_pred_lr = classifier.predict(X_test)
y_pred_prob_lr = classifier.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_pred_prob_lr)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, linewidth=2, color='darkorange')
plt.plot([0, 1], [0, 1], 'r--')
plt.title('ROC Curve of LOGISTIC REGRESSION')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.show()

cm = confusion_matrix(y_test, y_pred_lr, labels=classifier.classes_)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classifier.classes_).plot()
plt.show()

print('[INFO] evaluating after training...')
print('!!Confusion Matrix Report for Logistic Regression!!')
print(classification_report(y_test, y_pred_lr))
print('Accuracy_score:', '{:.4%}'.format(accuracy_score(y_test, y_pred_lr)))
print('ROC AUC Score:', '{:.4%}'.format(roc_auc_score(y_test, y_pred_prob_lr)))

## 9. K-Fold Cross Validation (20 splits)

In [ ]:
from sklearn import model_selection
from sklearn.model_selection import KFold

kfold = model_selection.KFold(n_splits=20, shuffle=True)

results_kfold_lr = model_selection.cross_val_score(classifier, X, y, cv=kfold)
print(f'Logistic Regression scores: {results_kfold_lr}')
print(f'Average: {"{:.4%}".format(results_kfold_lr.mean())}')

results_kfold_rf = model_selection.cross_val_score(rf_clf, X, y, cv=kfold)
print(f'Random Forest scores: {results_kfold_rf}')
print(f'Average: {"{:.4%}".format(results_kfold_rf.mean())}')

results_kfold_xgb = model_selection.cross_val_score(xgb, X, y, cv=kfold)
print(f'XGBoost scores: {results_kfold_xgb}')
print(f'Average: {"{:.4%}".format(results_kfold_xgb.mean())}')

results_kfold_dt = model_selection.cross_val_score(clf, X, y, cv=kfold)
print(f'Decision Tree scores: {results_kfold_dt}')
print(f'Average: {"{:.4%}".format(results_kfold_dt.mean())}')

## 10. Sample Prediction

In [ ]:
age = 80
avg_glucose_level = 105.92
bmi = 32.5
gender_Male = 1
ever_married_Yes = 1
work_type_Never_worked = 0
work_type_Private = 1
work_type_Self_employed = 0
work_type_children = 0
Residence_type_Urban = 0
smoking_status_formerly_smoked = 0
smoking_status_never_smoked = 1
smoking_status_smokes = 0
hypertension_1 = 0
heart_disease_1 = 1

input_features = [age, avg_glucose_level, bmi,
                  gender_Male, hypertension_1, heart_disease_1,
                  ever_married_Yes, work_type_Never_worked,
                  work_type_Private, work_type_Self_employed,
                  work_type_children, Residence_type_Urban,
                  smoking_status_formerly_smoked,
                  smoking_status_never_smoked, smoking_status_smokes]

features_name = ['age', 'avg_glucose_level', 'bmi',
                 'gender_Male', 'hypertension_1', 'heart_disease_1',
                 'ever_married_Yes', 'work_type_Never_worked',
                 'work_type_Private', 'work_type_Self_employed',
                 'work_type_children', 'Residence_type_Urban',
                 'smoking_status_formerly_smoked',
                 'smoking_status_never_smoked', 'smoking_status_smokes']

df_pred = pd.DataFrame([np.array(input_features)], columns=features_name)
prediction = rf_clf.predict(df_pred)[0]

if prediction == 1:
    print('Result: Cerebrovascular Accident Detected')
elif prediction == 0:
    print('Result: Cerebrovascular Accident Not Detected')
else:
    print('**Insufficient Information**')

## 11. Save Model

In [ ]:
import pickle
import os

os.makedirs('../models', exist_ok=True)
with open('../models/model.pickle', 'wb') as f:
    pickle.dump(rf_clf, f)
print('Model saved to models/model.pickle')